### RAG Piplines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\user\AppData\Local\Temp\ipykernel_121524\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files process")
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file type"] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error : {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdfs_documents = process_all_pdfs("../data")

Found 2 PDF files process
Processing ..\data\pdf\Faculty Meeting - 22 October 26.pdf...
 Loaded 18 pages
Processing ..\data\pdf\LMD - Examination and Quality Control - Intro.pdf...
 Loaded 11 pages

Total documents loaded: 29


In [ ]:
all_pdfs_documents

In [3]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show exmaple of a chunk
    if split_docs:
        print(f"Example chunk:")
        print(f"Context: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [4]:
chunks = split_documents(all_pdfs_documents)
chunks

Split 29 documents into 29 chunks
Example chunk:
Context: Institute for Banking Studies
CENTER FOR
BANKING STUDIES
SEARCHES FOR EXCELLENCECENTER FOR
BANKING STUDIES
SEARCHES FOR EXCELLENCE
Faculty Meeting 
Preparation for Semester 1 2026 - Faculty Members
Pr...
Metadata: {'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'creationdate': '2026-10-23T08:44:44+07:00', 'title': 'PowerPoint Presentation', 'author': 'Kry  Yokhun', 'subject': 'Une supervision locale intégrée dans l’Europe (Congrès OCBF 2013 - Amsterdam)', 'moddate': '2026-10-23T08:44:44+07:00', 'source': '..\\data\\pdf\\Faculty Meeting - 22 October 26.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Faculty Meeting - 22 October 26.pdf', 'file type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'creationdate': '2026-10-23T08:44:44+07:00', 'title': 'PowerPoint Presentation', 'author': 'Kry  Yokhun', 'subject': 'Une supervision locale intégrée dans l’Europe (Congrès OCBF 2013 - Amsterdam)', 'moddate': '2026-10-23T08:44:44+07:00', 'source': '..\\data\\pdf\\Faculty Meeting - 22 October 26.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Faculty Meeting - 22 October 26.pdf', 'file type': 'pdf'}, page_content='Institute for Banking Studies\nCENTER FOR\nBANKING STUDIES\nSEARCHES FOR EXCELLENCECENTER FOR\nBANKING STUDIES\nSEARCHES FOR EXCELLENCE\nFaculty Meeting \nPreparation for Semester 1 2026 - Faculty Members\nPrepared by: Lecturing Management Division\nVenue: Lecturer Theater 2, IBS\nDate: 23 October 2026'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'creationdate': '2026-10-23T08:44:44+07:00'

### Embedding and VectorStoreDB


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\project\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name : HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts : List of strings to embed.

        Returns:
            numpy array of embeddings with shap (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3465.12it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\user\AppData\Local\Temp\ipykernel_121524\4059542019.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 232


In [8]:
chunks

[Document(metadata={'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'creationdate': '2026-10-23T08:44:44+07:00', 'title': 'PowerPoint Presentation', 'author': 'Kry  Yokhun', 'subject': 'Une supervision locale intégrée dans l’Europe (Congrès OCBF 2013 - Amsterdam)', 'moddate': '2026-10-23T08:44:44+07:00', 'source': '..\\data\\pdf\\Faculty Meeting - 22 October 26.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'source_file': 'Faculty Meeting - 22 October 26.pdf', 'file type': 'pdf'}, page_content='Institute for Banking Studies\nCENTER FOR\nBANKING STUDIES\nSEARCHES FOR EXCELLENCECENTER FOR\nBANKING STUDIES\nSEARCHES FOR EXCELLENCE\nFaculty Meeting \nPreparation for Semester 1 2026 - Faculty Members\nPrepared by: Lecturing Management Division\nVenue: Lecturer Theater 2, IBS\nDate: 23 October 2026'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'creationdate': '2026-10-23T08:44:44+07:00'

In [9]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 29 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]


Generated embeddings with shape: (29, 384)
Adding 29 documents to vector store...
Successfully added 29 documents to vector store
Total documents in collection: 261


### Retriever Pipeline From VectorStore

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [11]:
rag_retriever

In [12]:
rag_retriever.retrieve("What is the Quality Control")

Retrieving documents for query: 'What is the Quality Control'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 115.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3ce01b04_25',
  'content': 'Quality Control\nQuality Control\nDuring the semester, students have the opportunity to complete the survey\ntwice, at mid-semester and the end of the semester, to help identify\nimprovement areas for teaching performance. The LMD team will facilitate\nthese surveys.',
  'metadata': {'content_length': 262,
   'title': 'CBS',
   'author': 'Michel',
   'page': 7,
   'source_file': 'LMD - Examination and Quality Control - Intro.pdf',
   'total_pages': 11,
   'moddate': '2026-05-22T10:36:06+07:00',
   'creator': 'Microsoft® PowerPoint® LTSC',
   'page_label': '8',
   'file type': 'pdf',
   'creationdate': '2026-05-22T10:36:06+07:00',
   'producer': 'Microsoft® PowerPoint® LTSC',
   'doc_index': 25,
   'source': '..\\data\\pdf\\LMD - Examination and Quality Control - Intro.pdf',
   'subject': 'Une supervision locale intégrée dans l’Europe (Congrès OCBF 2013 - Amsterdam)'},
  'similarity_score': 0.04583275318145752,
  'distance': 0.9541672468185425,
 

### Integration Vectordb Context pipline With LLM output

In [13]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [14]:
answer=rag_simple("tell me about the retake exam?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'tell me about the retake exam?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 102.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The retake exam is a single extra chance for students who didn’t pass. Its key rules are:

* **Score cap:** The highest recorded score for a retake is **60 / 100**.  
* **If the raw score is > 60** (e.g., 65, 70), it is reduced to **60**.  
* **If the raw score is 50‑59**, the score is kept as‑is.  
* **If the raw score is < 50**, the student must repeat the failing subjects and go through the normal promotion process again.


### Enhanced RAG Pipline Features

In [15]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("tell me about Attendance & Leave", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'tell me about Attendance & Leave'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 114.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Attendance & Leave – Key Points**

- **Purpose:** Allows lecturers to request leave when unforeseen or personal matters prevent them from teaching a scheduled session.  
- **How to request:**  
  1. Fill out the official **Leave Request Form**.  
  2. Email it to **lmd@ibs.edu.kh** with the subject line **“Leave Request_ Mr./Ms. [Name]”**.  
- **Notice period:**  
  - **Preferred:** Notify the LMD at least **2 days before** the intended absence.  
  - **Emergency (same‑day) situations:** Call or message the LMD team as soon as you know you cannot attend.  
- **After a leave is approved:**  
  - Arrange a **make‑up class** date.  
  - Inform both the LMD and the affected **students** of the new schedule.  

These steps ensure smooth coordination and minimal disruption to the teaching schedule.
Sources: [{'source': 'Faculty Meeting - 22 October 26.pdf', 'page': 7, 'score': 0.22241830825805664, 'preview': "Institute for Banking Studies 7\nAttendance & Leave (Cont')\nRequesting L

### Advance RAG Pipeline Features

In [21]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("tell me about Attendance & Leave", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'tell me about Attendance & Leave'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Institute for Banking Studies 7
Attendance & Leave (Cont')
Requesting Leave
- We understand that unforeseen circumstances or personal matters may occasionally 
prevent lecturers from attending their scheduled sessions.
• Contact: Complete the official

 Leave Request Form and submit it to LMD.
via lmd@ibs.edu.kh. (Subject : Leave Request_Mr.xxxx)
* Notice period: 
- Ideally, inform LMD at least 2 days in advance whenever possible.
- For Emergency Situations (same day), please call or message to our team directly as 
soon as you are aware you cannot attend.
*** (If Lecturers requested leave, Please kindly arrange the date for make up class and inform 
us & your respective students.)

Institute for Banking Studies 7
Attendance & Leave (Cont')
Requesting Leave
- We understand that unforeseen circumstances or personal matters may occasionally 
prevent lecturers from attending their scheduled sessions.
• Contact: Complete the official Leave Request Form and submit it to LMD.
via lmd@ibs.edu.kh. (Subject : Leave Request_Mr.xxxx)
* Notice period: 
- Ideally, inform LMD at least 2 days in advance whenever possible.
- For Emergency Situations (same day), please call or message to our team directly as 
soon as you are aware you cannot attend.
